DATA LOADING AND PREPROCESSING

In [1]:
!pip install -q datasets gensim transformers scikit-learn nltk bs4


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.2 MB/s eta 0:00:00


In [2]:
import re
import random
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import nltk
nltk.download('punkt')

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [3]:
dataset = load_dataset("imdb")

train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts = dataset['test']['text']
test_labels = dataset['test']['label']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
def clean_text(text):
    text = text.lower()
    text = BeautifulSoup(text, "html.parser").get_text()  # remove HTML
    text = re.sub(r"[^\w\s]", "", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [5]:
train_clean = [clean_text(t) for t in train_texts]
test_clean  = [clean_text(t) for t in test_texts]


In [6]:
#tokenization

In [7]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
train_tokens = [word_tokenize(t) for t in train_clean]
test_tokens  = [word_tokenize(t) for t in test_clean]



VECTORIZATION AND FEATURE REPRESENTATION

TF IDF model

In [9]:
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(train_clean)
X_test_tfidf  = tfidf.transform(test_clean)


In [10]:
tfidf_clf = LinearSVC()
tfidf_clf.fit(X_train_tfidf, train_labels)

tfidf_preds = tfidf_clf.predict(X_test_tfidf)


Word2VEC

In [11]:
w2v_model = Word2Vec(sentences=train_tokens,
                     vector_size=100,
                     window=5,
                     min_count=2,
                     workers=4,
                     seed=42)


In [12]:
def avg_vector(tokens, model, dim=100):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_w2v = np.array([avg_vector(t, w2v_model) for t in train_tokens])
X_test_w2v  = np.array([avg_vector(t, w2v_model) for t in test_tokens])


In [13]:
w2v_clf = LogisticRegression(max_iter=1000)
w2v_clf.fit(X_train_w2v, train_labels)

w2v_preds = w2v_clf.predict(X_test_w2v)


**BERT**

In [14]:
print(type(train_texts))
print(type(train_texts[0]))
print(train_texts[0])


<class 'datasets.arrow_dataset.Column'>
<class 'str'>
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, 

In [15]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_batch(texts):
    return tokenizer(texts, padding=True, truncation=True, max_length=256)

train_texts = list(train_texts)
test_texts  = list(test_texts)

train_encodings = tokenize_batch(train_texts)
test_encodings  = tokenize_batch(test_texts)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [16]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset  = IMDbDataset(test_encodings, test_labels)


In [17]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    seed=42,
    load_best_model_at_end=True
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()
bert_results = trainer.evaluate()


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss
1,0.253563,0.247620
2,0.124432,0.281480


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

EVALUATION

In [22]:
def evaluate(true, preds):
    precision, recall, f1, _ = precision_recall_fscore_support(true, preds, average='binary')
    acc = accuracy_score(true, preds)
    return acc, precision, recall, f1

tfidf_metrics = evaluate(test_labels, tfidf_preds)
w2v_metrics   = evaluate(test_labels, w2v_preds)
preds = trainer.predict(test_dataset)
y_pred = np.argmax(preds.predictions, axis=1)

bert_metrics = evaluate(test_labels, y_pred)


COMPARISON AND ANALYSIS

In [24]:
results = pd.DataFrame({
    "Model": ["TF-IDF + SVM", "Word2Vec + LR", "BERT"],
    "Accuracy": [tfidf_metrics[0], w2v_metrics[0], bert_metrics[0]],
    "Precision": [tfidf_metrics[1], w2v_metrics[1], bert_metrics[1]],
    "Recall": [tfidf_metrics[2], w2v_metrics[2], bert_metrics[2]],
    "F1-Score": [tfidf_metrics[3], w2v_metrics[3], bert_metrics[3]],
})

results


,Model,Accuracy,Precision,Recall,F1-Score
0,TF-IDF + SVM,0.88812,0.889210,0.88672,0.887963
1,Word2Vec + LR,0.80456,0.804901,0.80400,0.804450
2,BERT,0.90216,0.953537,0.84552,0.896286


Traditional and neural text representations show clear performance differences on sentiment classification. The TF-IDF model, despite being a simple bag-of-words approach, achieved strong baseline performance due to its ability to capture important keywords and bigrams that strongly correlate with sentiment. It is computationally efficient and easy to train, making it a practical choice when resources are limited.

The Word2Vec model provided semantic understanding by embedding words into dense vectors. However, averaging word vectors removes word order and syntactic structure, which are crucial for sentiment tasks involving negation and nuanced phrasing. As a result, it typically performed worse than TF-IDF despite being more computationally expensive to train.

The BERT-based model achieved the best overall performance. Because BERT is a contextual language model, it understands how word meaning changes depending on surrounding words. This allows it to correctly interpret complex expressions such as sarcasm or negations like “not bad.” However, this improved accuracy comes at the cost of significantly higher computational requirements, longer training times, and greater memory usage.

In conclusion, while BERT provides state-of-the-art accuracy, TF-IDF remains a strong baseline for fast and efficient sentiment analysis. Word2Vec offers semantic richness but is less effective without more advanced aggregation strategies. The best choice depends on whether priority is accuracy or computational efficiency.

In [ ]:
print(bert_results.keys())
print(bert_results)

dict_keys(['eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'epoch'])
{'eval_loss': 0.2472151219844818, 'eval_runtime': 378.494, 'eval_samples_per_second': 66.051, 'eval_steps_per_second': 4.13, 'epoch': 2.0}
